# 中心极限定理的模拟

> 对应代码：`MathStatsCode/code_in_notes/Chap.8/simulate_CLT.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebook/PyStata.md`。

**实验目的**：从双峰混合正态分布中抽样，展示样本量2/3/4/10/100/1000时样本均值的分布逐步逼近正态分布。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。

首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.8`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。


In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.8")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

`clear all` 清空全部（含数据框与矩阵），`set more off`、`set seed 8855` 固定随机数种子。


In [ ]:
%%stata
clear all
set more off
set seed 8855


**混合正态分布**

构造一个**双峰总体**：`set obs 1000`，`gen d=runiform()>=0.5` 生成等概率的0/1指示变量；`gen x=d*rnormal(3,1)+(1-d)*rnormal(-3,1)`——以d为开关，一半观测来自 N(3,1)、另一半来自 N(-3,1)，故 x 服从均值为0的双峰混合正态分布（这使“总体不是正态”这一条件更醒目）；`hist x` 画直方图并以 `graph export` 导出（simulate_CLT_mix.pdf），可以看到明显的两个峰。


In [ ]:
%%stata
set obs 1000
gen d=runiform()>=0.5
gen x=d*rnormal(3,1)+(1-d)*rnormal(-3,1)
hist x


**模拟**

这一大段用 `foreach` 循环模拟不同样本量下**样本均值的分布**：对 N=2,3,4,10,100,1000 各做一次模拟——`frame create mean\`N' m` 为每个 N 新建结果数据框；`forvalues j=1/2000` 循环2000次：每次清空数据、生成 N 个混合正态观测、求均值、`frame post` 写入结果；`saving(simulate_CLT\`N'.gph)` 把每个 N 的均值直方图存成图形文件（N=2时直方图呈双峰，N 增大后逐渐变成钟形）；最后 `graph combine` 用保存的 .gph 文件把6个子图排成3列合并，`graph export` 导出。整个实验说明：即使总体严重非正态，样本量足够大时样本均值仍近似正态——即**中心极限定理**。


In [ ]:
%%stata
local graphs ""
foreach N in 2 3 4 10 100 1000{
	frame create mean`N' m
	frame mean`N': label variable m "样本均值"
	forvalues j=1/2000{
		qui{
            clear
    	    set obs `N'
    	    gen d=runiform()>=0.5
    	    gen x=d*rnormal(3,1)+(1-d)*rnormal(-3,1)
    	    su x
    	    frame post mean`N' (r(mean))
        }
	}
	frame mean`N': hist m, normal saving(simulate_CLT`N', replace) title("N=`N'")
	local graphs "`graphs' simulate_CLT`N'.gph"
}
graph combine `graphs', col(3)
